### 16. I denna uppgift ska du göra ett komplett ML-flöde där du modellerar diamantpriser som finns tillgängliga i datasetet "diamonds.csv". Du kan läsa mer om datasetet här: [https://www.kaggle.com/datasets/shivam2503/diamonds](https://www.kaggle.com/datasets/shivam2503/diamonds)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
diamonds = pd.read_csv("../../data/diamonds.csv")

In [3]:
print(diamonds.head(10))
print(f"\nStorlek: {diamonds.shape}")
print(f"\nInfo:")
diamonds.info()
print(f"\nSaknade värden: {diamonds.isnull().sum().sum()}")

   Unnamed: 0  carat        cut color clarity  depth  table  price     x  \
0           1   0.23      Ideal     E     SI2   61.5   55.0    326  3.95   
1           2   0.21    Premium     E     SI1   59.8   61.0    326  3.89   
2           3   0.23       Good     E     VS1   56.9   65.0    327  4.05   
3           4   0.29    Premium     I     VS2   62.4   58.0    334  4.20   
4           5   0.31       Good     J     SI2   63.3   58.0    335  4.34   
5           6   0.24  Very Good     J    VVS2   62.8   57.0    336  3.94   
6           7   0.24  Very Good     I    VVS1   62.3   57.0    336  3.95   
7           8   0.26  Very Good     H     SI1   61.9   55.0    337  4.07   
8           9   0.22       Fair     E     VS2   65.1   61.0    337  3.87   
9          10   0.23  Very Good     H     VS1   59.4   61.0    338  4.00   

      y     z  
0  3.98  2.43  
1  3.84  2.31  
2  4.07  2.31  
3  4.23  2.63  
4  4.35  2.75  
5  3.96  2.48  
6  3.98  2.47  
7  4.11  2.53  
8  3.78  2.49  
9  

## Rengör data

Tar bort alla diamanter som är mindre eller lika med 0 mm i storlek (bredd, längd, höjd)


In [4]:
diamanter_x_noll = len(diamonds[diamonds["x"] <= 0])
diamanter_y_noll = len(diamonds[diamonds["y"] <= 0])
diamanter_z_noll = len(diamonds[diamonds["z"] <= 0])

print(f"Diamanter med bredd (x) <= 0 mm:   {diamanter_x_noll} st")
print(f"Diamanter med längd (y) <= 0 mm:   {diamanter_y_noll} st")
print(f"Diamanter med höjd (z) <= 0 mm:    {diamanter_z_noll} st")
print(
    f"\nTotalt:                            {diamanter_x_noll + diamanter_y_noll + diamanter_z_noll} st"
)

problem_diamanter = diamonds[
    (diamonds["x"] <= 0) | (diamonds["y"] <= 0) | (diamonds["z"] <= 0)
]
print(f"Antal unika:                       {len(problem_diamanter)} st")
diamonds_clean = diamonds[
    (diamonds["x"] > 0) & (diamonds["y"] > 0) & (diamonds["z"] > 0)
]

print(f"Diamanter innan rengöring:         {len(diamonds)} st")
print(f"Diamanter efter rengöring:         {len(diamonds_clean)} st")
print(f"Borttagna:                         {len(diamonds) - len(diamonds_clean)} st")

diamonds = diamonds_clean

Diamanter med bredd (x) <= 0 mm:   8 st
Diamanter med längd (y) <= 0 mm:   7 st
Diamanter med höjd (z) <= 0 mm:    20 st

Totalt:                            35 st
Antal unika:                       20 st
Diamanter innan rengöring:         53940 st
Diamanter efter rengöring:         53920 st
Borttagna:                         20 st


In [5]:
# Ta bort Unnamed kolumn
if "Unnamed: 0" in diamonds.columns:
    diamonds = diamonds.drop(columns=["Unnamed: 0"])

# Dummy variables för kategoriska kolumner
diamonds = pd.get_dummies(
    diamonds, columns=["cut", "color", "clarity"], drop_first=True
)

print(diamonds.head())

   carat  depth  table  price     x     y     z  cut_Good  cut_Ideal  \
0   0.23   61.5   55.0    326  3.95  3.98  2.43     False       True   
1   0.21   59.8   61.0    326  3.89  3.84  2.31     False      False   
2   0.23   56.9   65.0    327  4.05  4.07  2.31      True      False   
3   0.29   62.4   58.0    334  4.20  4.23  2.63     False      False   
4   0.31   63.3   58.0    335  4.34  4.35  2.75      True      False   

   cut_Premium  ...  color_H  color_I  color_J  clarity_IF  clarity_SI1  \
0        False  ...    False    False    False       False        False   
1         True  ...    False    False    False       False         True   
2        False  ...    False    False    False       False        False   
3         True  ...    False     True    False       False        False   
4        False  ...    False    False     True       False        False   

   clarity_SI2  clarity_VS1  clarity_VS2  clarity_VVS1  clarity_VVS2  
0         True        False        False     

## Train, Validation, and Test Set

In [6]:
X = diamonds.drop(columns=["price"])
y = diamonds["price"]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42
)

print(f"Träning: {len(X_train)}")
print(f"Validering: {len(X_val)}")
print(f"Test: {len(X_test)}")

Träning: 32352
Validering: 10784
Test: 10784


## Träna modeller

In [7]:
# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_val)
lr_rmse = np.sqrt(mean_squared_error(y_val, lr_pred))
print(f"Linear Regression RMSE: {lr_rmse:.2f}")

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_val)
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_pred))
print(f"Random Forest RMSE: {rf_rmse:.2f}")

Linear Regression RMSE: 1165.46
Random Forest RMSE: 683.07


## Välj bästa och testa

In [8]:
if rf_rmse < lr_rmse:
    best_model = RandomForestRegressor(n_estimators=100, random_state=42)
    best_name = "Random Forest"
else:
    best_model = LinearRegression()
    best_name = "Linear Regression"

print(f"Vinnare: {best_name}")

best_model.fit(X_train_full, y_train_full)
y_pred_test = best_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))

print(f"\nRMSE på test: {test_rmse:.2f}")

Vinnare: Random Forest

RMSE på test: 589.42


## Träna modellen på hela datasetet

In [9]:
if best_name == "Random Forest":
    final_model = RandomForestRegressor(n_estimators=100, random_state=42)
else:
    final_model = LinearRegression()

final_model.fit(X, y)

final_pred = final_model.predict(X)
final_rmse = np.sqrt(mean_squared_error(y, final_pred))

print(f"  Hela modellen är tränad på {len(X)} diamanter")
print(f"  RMSE på hela dataset: {final_rmse:.2f}")

  Hela modellen är tränad på 53920 diamanter
  RMSE på hela dataset: 224.22
